In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import pymysql
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# ======================
# 数据库连接
# ======================
def get_df(db, sql):
    conn = pymysql.connect(
        host="localhost",
        user="root",
        password="123456",
        database=db,
        charset="utf8mb4"
    )
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

# 检查连接
try:
    get_df("yx_ai_feature", "SELECT 1")
    print("✓ 数据库连接成功！")
except Exception as e:
    print(f"✗ 数据库连接失败: {e}")
    exit()

# 读取数据（包含 ip_count）
df = get_df("yx_ai_feature", """
    SELECT scan_count, time_variance, location_variance, device_count, ip_count, is_reused
    FROM reuse_pattern
""")
print(f"✓ 读取数据: {len(df)} 条记录")
print(f"✓ 类别分布: {df['is_reused'].value_counts().to_dict()}")

# ======================
# 数据预处理
# ======================
FEATURES = ["scan_count", "time_variance", "location_variance", "device_count", "ip_count"]
X = df[FEATURES].values
y = df["is_reused"].values

# 计算类别权重（解决数据不平衡）
class_counts = np.bincount(y)
class_weights = torch.tensor([class_counts[0]/len(y), class_counts[1]/len(y)], dtype=torch.float32)
print(f"\n✓ 类别权重: 负样本={class_weights[0]:.4f}, 正样本={class_weights[1]:.4f}")

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 分层划分训练测试集（保持类别比例）
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# 转 PyTorch 张量
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# ======================
# 教师模型（增强版双通道）
# 包含Transformer编码器 + 图分支
# ======================
class TeacherModel(nn.Module):
    def __init__(self, in_dim=5, hidden_dim=32):
        super().__init__()
        self.time_proj = nn.Linear(in_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            dim_feedforward=hidden_dim * 2,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.graph_branch = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim)
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        seq = self.time_proj(x).unsqueeze(1)
        seq_out = self.transformer(seq).squeeze(1)
        graph_out = self.graph_branch(x)
        fused = torch.cat([seq_out, graph_out], dim=1)
        return self.head(fused)

# ======================
# 学生模型（优化版）
# ======================
class StudentModel(nn.Module):
    def __init__(self, in_dim=5):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.2),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.fc(x)

# ======================
# 训练教师模型
# 使用加权损失解决数据不平衡
# ======================
teacher = TeacherModel()
opt = torch.optim.AdamW(teacher.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

print("\n--- 训练教师模型 ---")
for e in range(50):
    teacher.train()
    logits = teacher(X_train)
    loss = criterion(logits, y_train)
    opt.zero_grad()
    loss.backward()
    opt.step()
    scheduler.step()

    if (e+1) % 10 == 0:
        teacher.eval()
        with torch.no_grad():
            val_logits = teacher(X_test)
            val_pred = torch.argmax(val_logits, dim=1)
            val_acc = (val_pred == y_test).float().mean().item()
        print(f"第{e+1}轮 | 损失: {loss.item():.4f} | 验证准确率: {val_acc:.4f}")

# ======================
# 知识蒸馏训练学生模型
# 混合损失：监督损失 + 蒸馏损失
# ======================
student = StudentModel()
opt = torch.optim.AdamW(student.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
ce_criterion = nn.CrossEntropyLoss(weight=class_weights)
kl_criterion = nn.KLDivLoss(reduction="batchmean")

print("\n--- 知识蒸馏训练学生模型 ---")
for e in range(50):
    student.train()
    teacher.eval()

    with torch.no_grad():
        t_out = teacher(X_train)
    s_out = student(X_train)

    # 混合损失：监督损失 + 蒸馏损失
    ce_loss = ce_criterion(s_out, y_train)
    kl_loss = kl_criterion(F.log_softmax(s_out / 2, dim=1), F.softmax(t_out / 2, dim=1)) * (2.0 ** 2)
    loss = 0.6 * ce_loss + 0.4 * kl_loss

    opt.zero_grad()
    loss.backward()
    opt.step()
    scheduler.step()

    if (e+1) % 10 == 0:
        student.eval()
        with torch.no_grad():
            val_logits = student(X_test)
            val_pred = torch.argmax(val_logits, dim=1)
            val_acc = (val_pred == y_test).float().mean().item()
        print(f"第{e+1}轮 | 损失: {loss.item():.4f} | 验证准确率: {val_acc:.4f}")

# ======================
# 模型评估
# ======================
print("\n" + "="*50)
print("          模型性能评估")
print("="*50)

student.eval()
with torch.no_grad():
    train_logits = student(X_train)
    test_logits = student(X_test)
    train_pred = torch.argmax(train_logits, dim=1)
    test_pred = torch.argmax(test_logits, dim=1)
    test_proba = torch.softmax(test_logits, dim=1)[:, 1]

# 计算指标
def evaluate(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else "N/A"
    cm = confusion_matrix(y_true, y_pred)
    return acc, prec, recall, f1, auc, cm

# 训练集评估
train_acc, train_prec, train_recall, train_f1, _, _ = evaluate(y_train.numpy(), train_pred.numpy())
print(f"【训练集】")
print(f"  准确率: {train_acc:.4f}")
print(f"  F1分数: {train_f1:.4f}")

# 测试集评估
test_acc, test_prec, test_recall, test_f1, test_auc, cm = evaluate(y_test.numpy(), test_pred.numpy(), test_proba.numpy())
print(f"\n【测试集】")
print(f"  准确率: {test_acc:.4f}")
print(f"  精确率: {test_prec:.4f}")
print(f"  召回率: {test_recall:.4f}")
print(f"  F1分数: {test_f1:.4f}")
print(f"  AUC: {test_auc:.4f}")
print("\n【混淆矩阵】")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")
print(f"  漏检率: {cm[1,0]/(cm[1,0]+cm[1,1]):.4f}")

# ======================
# 阈值调优（针对召回率优化）
# ======================
print("\n" + "="*50)
print("         阈值调优结果")
print("="*50)

best_f1 = 0
best_threshold = 0.5
for threshold in np.arange(0.1, 0.9, 0.05):
    threshold_pred = (test_proba.numpy() >= threshold).astype(int)
    f1 = f1_score(y_test.numpy(), threshold_pred)
    recall = recall_score(y_test.numpy(), threshold_pred)
    prec = precision_score(y_test.numpy(), threshold_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
    print(f"阈值={threshold:.2f} | F1={f1:.4f} | 召回率={recall:.4f} | 精确率={prec:.4f}")

print(f"\n✓ 最佳阈值: {best_threshold:.2f}，对应F1: {best_f1:.4f}")

# 使用最佳阈值评估
best_pred = (test_proba.numpy() >= best_threshold).astype(int)
best_cm = confusion_matrix(y_test.numpy(), best_pred)
print(f"\n【最佳阈值评估结果】")
print(f"  准确率: {accuracy_score(y_test.numpy(), best_pred):.4f}")
print(f"  精确率: {precision_score(y_test.numpy(), best_pred):.4f}")
print(f"  召回率: {recall_score(y_test.numpy(), best_pred):.4f}")
print(f"  F1分数: {best_f1:.4f}")
print(f"\n【混淆矩阵】")
print(f"  TN={best_cm[0,0]}  FP={best_cm[0,1]}\n  FN={best_cm[1,0]}  TP={best_cm[1,1]}")
print(f"  漏检率: {best_cm[1,0]/(best_cm[1,0]+best_cm[1,1]):.4f}")

# ======================
# 导出 ONNX 模型
# ======================
student.eval()
torch.onnx.export(
    student,
    torch.randn(1, 5),
    "qs_risk_model.onnx",
    input_names=["features"],
    output_names=["logits"],
    dynamic_axes={"features": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13,
    export_params=True,
    do_constant_folding=True,
    save_as_external_data=False,
    external_data_filename=""
)
print("\n✓ 模型已导出为 qs_risk_model.onnx")
print(f"✓ 特征顺序: {FEATURES}")
print(f"✓ 推荐分类阈值: {best_threshold:.2f}")
print(f"✓ 最佳F1分数: {best_f1:.4f}")